# Screen Macro -- Object Detection Training (RF-DETR, Segmentation)

Segmentation-head variant of `train_object_detector.ipynb`, trained on a polygon-labeled dataset (Screen Macro's "Label Frames" -> Polygon mode, see the main game-bot repo's `ai_tab.py`) instead of a box-labeled one, on the theory that real per-pixel mask supervision gives the model a tighter, more reliable sense of the object's actual shape than a box alone -- worth trying since a first head-to-head YOLO-vs-RF-DETR box run on a small (20-image) dataset showed a real reliability gap worth investigating further (see the main game-bot repo's `docs/cv-object-detection-investigation.md`).

**Exports the exact same plain `[N,6]` box-only ONNX contract as `train_object_detector.ipynb`'s box-only fallback path** -- no mask output, no app-side changes needed. Segmentation is training-time supervision only, meant to potentially improve box localization; `onnx_detector.py` and the app's "+ Detect Model" action don't need to know or care this model was ever trained with masks. Box center is the click fallback (same as any keypoint-less model) -- **this notebook does not predict a keypoint**, since rf-detr's segmentation head and keypoint-preview head are mutually exclusive, hard-enforced by the library itself (confirmed directly against the pinned `1.9.4` source -- `models/postprocess.py` raises if both are configured on the same model at once; see step 4's markdown).

**Requires a polygon-labeled dataset.** Every labeled object needs a `"polygon"` field in `labels.json` (Screen Macro's Label Frames -> Polygon mode) -- a box-only dataset has nothing for the segmentation loss to train against. If some object somehow lacks one (shouldn't happen given the app enforces one annotation shape per whole dataset), this notebook falls back to a rectangular pseudo-polygon derived from that object's box rather than silently zeroing its mask supervision -- see step 3's markdown for why that specific failure mode matters here.

**Colab (default, manual):**
1. `Runtime` -> `Change runtime type` -> select a **GPU** (T4 is fine, free tier).
2. `Runtime` -> `Run all`.
3. Wait for a "Choose files" button to appear below step 2's cell (can take a minute -- it installs dependencies first), click it, and upload the `..._dataset.zip` file Screen Macro produced (must be a polygon-labeled dataset, see above).
4. Everything else runs on its own -- training, export, and an optional sanity check (step 6, safe to skip) -- then `best.onnx` downloads automatically at the end (or copies to Google Drive instead, see step 7). Import it back into the app via the AI tab's "Import Model...".

**Kaggle (opt-in, scripted):** same as the other two notebooks in this repo -- the app's `cv_training.py` pushes this notebook as a Kaggle kernel with the dataset attached, polls, and pulls `best.onnx` back automatically.

**Status: not yet run against a real dataset.** Code-complete against rfdetr's documented segmentation API, confirmed directly against the pinned `1.9.4` source rather than assumed (see each step's own markdown for what was actually verified and how) -- but exactly like every other notebook in this repo's history, expect it to need at least one round of real-Colab-run debugging before treating it as proven.

## 1. Install dependencies

Same pinned versions as `train_object_detector.ipynb` -- an unpinned install risks a future upstream release silently changing behavior underneath this notebook, exactly as happened repeatedly to that notebook's own history.

In [ ]:
!pip install -q "rfdetr[train,loggers,augment]==1.9.4" "onnx==1.22.0" "onnxruntime==1.29.0"

### Optional: check what hardware this session got

Purely informational. `BATCH_SIZE = "auto"` in step 4 already sizes itself to whatever GPU you're connected to, free tier or paid -- and unlike `train_object_detector.ipynb`'s keypoint-preview path, this is actually safe to use unmodified for segmentation (see step 4's markdown).

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print('\nYour runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20:
    print('Not using a high-RAM runtime')
else:
    print('You are using a high-RAM runtime!')

## 2. Upload and unpack the dataset

Expects the zip Screen Macro's Label Frames (Polygon mode) produces: an `images/` folder of captured frames plus a `labels.json` of the form
`{"images_dir": "images", "annotation_type": "polygon", "labels": [{"image": "frame_001.png", "objects": [{"box": [x, y, w, h], "keypoint": [x, y], "polygon": [[x1, y1], [x2, y2], ...]}, ...]}, ...]}`
(frames with no entry in `labels` were skipped during labeling -- the object wasn't visible in them; each frame can list more than one object). The `keypoint` field is ignored by this notebook (see the intro above for why); `polygon` is what step 3's COCO conversion actually trains against.

In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

RAW_DIR = Path("dataset_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

# cv_training.py's push already uploaded the dataset zip as this kernel's one attached
# Kaggle Dataset -- mounted read-only under /kaggle/input, no interactive upload prompt
# needed the way Colab's files.upload() requires. Detected by whether a zip is actually
# there, not just os.path.exists("/kaggle/input") -- that directory exists (empty) on
# plain Colab too.
candidates = glob.glob("/kaggle/input/*/*.zip") + glob.glob("/kaggle/input/*.zip")
ON_KAGGLE = bool(candidates)
if ON_KAGGLE:
    zip_name = candidates[0]
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(RAW_DIR)

print("Extracted:", list(RAW_DIR.iterdir()))

## 3. Convert to COCO format (train/valid split)

One category (`"object"`), with a polygon-format `"segmentation"` per annotation instead of `train_object_detector.ipynb`'s `"keypoints"`. rfdetr's own COCO loader (`datasets/coco.py`, confirmed against the pinned `1.9.4` source) only decodes masks for an image at all if that image's *first* annotation has a `"segmentation"` key -- if it's missing, every object in that image silently gets an empty mask instead of raising, which would quietly corrupt training rather than fail loudly. To guarantee every annotation always has one, an object's real `"polygon"` is used when present, and a rectangular fallback (a box turned into its 4-corner polygon) is used for the rare object that somehow doesn't have one.

**No copy-paste synthetic-frame step here**, unlike `train_object_detector.ipynb`'s step 3 -- that step composites via a warped alpha mask, not a real polygon, and correctly rotating/scaling a polygon's own points (not just its bounding box) through the same transform is nontrivial enough to be its own separate piece of work. Training on real captured+labeled frames only for this first version, same choice `train_object_detector_yolo.ipynb` already made for its own first version.

Same 90/10 `VALID_FRACTION` split as the other notebooks; adjust if you captured a lot more images.

In [ ]:
import json, random
import cv2

VALID_FRACTION = 0.1
COCO_DIR = Path("dataset_coco")

with open(RAW_DIR / "labels.json") as f:
    raw = json.load(f)
images_dir = RAW_DIR / raw.get("images_dir", "images")
entries = raw["labels"]

random.seed(0)
random.shuffle(entries)
n_valid = max(1, int(len(entries) * VALID_FRACTION)) if len(entries) > 1 else 0
splits = {"valid": entries[:n_valid], "train": entries[n_valid:]}


def _segmentation_for(obj):
    '''Real polygon when Screen Macro's polygon labeling produced one; a rectangular
    fallback derived from the box otherwise -- see this step's markdown for why every
    annotation always needs SOME segmentation entry, not just the ones with a real
    polygon.'''
    polygon = obj.get("polygon")
    if polygon:
        return [[coord for point in polygon for coord in point]]
    bx, by, bw, bh = obj["box"]
    return [[bx, by, bx + bw, by, bx + bw, by + bh, bx, by + bh]]


for split_name, split_entries in splits.items():
    split_dir = COCO_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    images_out, annotations_out = [], []
    for i, e in enumerate(split_entries, start=1):
        src = images_dir / e["image"]
        img = cv2.imread(str(src))
        h, w = img.shape[:2]
        shutil.copy(src, split_dir / e["image"])
        images_out.append({"id": i, "file_name": e["image"], "width": w, "height": h})
        # Each frame can carry more than one labeled object -- one COCO annotation per
        # object, all sharing this frame's image_id.
        for obj in e["objects"]:
            bx, by, bw, bh = obj["box"]
            annotations_out.append({
                "id": len(annotations_out) + 1, "image_id": i, "category_id": 1,
                "bbox": [bx, by, bw, bh],
                "area": bw * bh,
                "iscrowd": 0,
                "segmentation": _segmentation_for(obj),
            })
    coco = {
        "images": images_out,
        "annotations": annotations_out,
        "categories": [{"id": 1, "name": "object", "supercategory": "object"}],
    }
    with open(split_dir / "_annotations.coco.json", "w") as f:
        json.dump(coco, f)
    print(f"{split_name}: {len(images_out)} image(s), {len(annotations_out)} annotation(s)")

## 4. Train

Trains `RFDETRSegNano()` -- rfdetr's real, currently-supported (non-deprecated) small segmentation-head variant, with its own pretrained checkpoint (`rf-detr-seg-nano.pt`) -- confirmed directly against the pinned `1.9.4` source (`rfdetr/variants.py`) rather than assumed to exist. `segmentation_head=True` is a hard architectural flag baked into this model family's own config (not a toggle on the plain `RFDETRNano` the box-only notebook's fallback path uses), and confirmed mutually exclusive with keypoint-preview mode at the library level (`models/postprocess.py` raises if both are configured on the same model at once) -- so this notebook never attempts keypoint prediction; the exported model gets its click point from the box center, same as any keypoint-less model.

**No fallback to box-only on failure**, unlike `train_object_detector.ipynb`'s keypoint-preview attempt -- `RFDETRSegNano` isn't a "try it, fall back if it doesn't work" experiment the way keypoint-preview's own auto-batch incompatibility made necessary; it's a real, supported model family, so a training failure here is worth seeing the real traceback for, not silently papering over by downgrading to the plain box-only notebook's role.

`BATCH_SIZE = "auto"` is safe here, unlike keypoint-preview -- confirmed directly against `training/auto_batch.py`'s synthetic-batch prober, which already builds a correctly-shaped dummy `"masks"` entry whenever `segmentation_head=True`, unlike its total lack of any keypoint-aware branch (the confirmed root cause of keypoint-preview's own `KeyError('keypoints')` history in `train_object_detector.ipynb`). No fixed-batch workaround needed.

**Augmentation** reuses the exact same `AUG_CONFIG` as `train_object_detector.ipynb` (mild brightness/contrast, small rotation, light blur/noise) -- that reasoning is about the augmentation itself, not whether segmentation or keypoint supervision is used, so it carries over unchanged.

**Early stopping's metric names differ for segmentation** -- confirmed against `training/trainer.py`: segmentation's monitored metrics are `val/segm_mAP_50_95`/`val/ema_segm_mAP_50_95`, not `train_object_detector.ipynb`'s box-only `val/mAP_50_95`/`val/ema_mAP_50_95`. Built-in `early_stopping=True` already branches on this correctly internally -- only the custom mAP-ceiling-shortcut callback below (ported from that notebook) needed its hardcoded metric keys updated to match, or it would silently never fire.

In [ ]:
from rfdetr import RFDETRSegNano

model = RFDETRSegNano()

EPOCHS = 200      # small dataset -- more epochs than a COCO-scale run, early stopping cuts it short once it plateaus
BATCH_SIZE = "auto"  # safe for segmentation -- see this step's markdown
EARLY_STOPPING_PATIENCE = 20     # stop once segm mAP@50:95 hasn't improved for this many epochs
EARLY_STOPPING_MIN_DELTA = 0.001 # smallest mAP improvement that still counts as progress

# Same reasoning as train_object_detector.ipynb's own AUG_CONFIG -- see this step's
# markdown. Requires the rfdetr[augment] extra (installed in step 1).
AUG_CONFIG = {
    "RandomBrightnessContrast": {"brightness_limit": 0.15, "contrast_limit": 0.15, "p": 0.4},
    "Rotate": {"limit": 10, "p": 0.3},
    "GaussianBlur": {"blur_limit": 3, "p": 0.25},
    "GaussNoise": {"std_range": (0.01, 0.04), "p": 0.25},
}

# Same mAP-ceiling-shortcut idea as train_object_detector.ipynb's own callback, with the
# metric keys updated for segmentation's own names -- see this step's markdown. Stops the
# instant both tracks hit segm mAP@50:95's 1.0 ceiling instead of waiting out the full
# EARLY_STOPPING_PATIENCE window afterward, since neither can ever register a further
# "improvement" past 1.0.
try:
    import pytorch_lightning as pl
    import rfdetr.training as _rfdetr_training

    class _StopAtMapCeiling(pl.Callback):
        _CEILING = 0.9999  # allow for float rounding just under the true 1.0 max

        def on_validation_epoch_end(self, trainer, pl_module):
            metrics = trainer.callback_metrics
            regular = metrics.get("val/segm_mAP_50_95")
            ema = metrics.get("val/ema_segm_mAP_50_95")
            if regular is None or ema is None:
                return
            if float(regular) >= self._CEILING and float(ema) >= self._CEILING:
                print(
                    f"Both regular ({float(regular):.4f}) and EMA ({float(ema):.4f}) segm "
                    "mAP@50:95 hit the ceiling -- stopping now instead of waiting out early "
                    "stopping's patience."
                )
                trainer.should_stop = True

    if not getattr(_rfdetr_training.build_trainer, "_ceiling_stop_installed", False):
        def _build_trainer_with_ceiling_stop(*args, _orig=_rfdetr_training.build_trainer, **kwargs):
            trainer = _orig(*args, **kwargs)
            trainer.callbacks.append(_StopAtMapCeiling())
            return trainer

        _build_trainer_with_ceiling_stop._ceiling_stop_installed = True
        _rfdetr_training.build_trainer = _build_trainer_with_ceiling_stop
except Exception as exc:
    print(f"Could not enable the mAP-ceiling stop shortcut ({exc}) -- early stopping will still work normally.")

try:
    model.train(
        dataset_dir=str(COCO_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE,
        early_stopping=True, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, aug_config=AUG_CONFIG,
    )
except Exception:
    import traceback
    print("Training failed. Full traceback:")
    traceback.print_exc()
    raise

## 5. Export to ONNX with the app's expected contract

Reuses `train_object_detector.ipynb`'s own `ExportWrapper`/export approach for its box-only fallback path almost unchanged -- confirmed directly against rfdetr's pinned source that this needs no real modification for a segmentation-head model. `forward_export` returns a 3-tuple `(outputs_coord, outputs_class, outputs_masks)` for a segmentation model -- the exact same shape as keypoint mode's `(outputs_coord, outputs_class, outputs_keypoints)`. `ExportWrapper` below only ever reads its trailing third element when built for keypoint mode; here it's simply never touched, and `PostProcess.forward` falls through its plain box/score/label path whenever `"pred_masks"` isn't in the dict it's handed (confirmed in the source, not assumed) -- so this cell is essentially `train_object_detector.ipynb`'s box-only fallback export cell with `RFDETRNano()` swapped for this notebook's already-trained segmentation model, nothing else different.

Same best-checkpoint-loading discipline as the other notebook (`model.train()` only leaves the last trained epoch's weights in memory, not necessarily the best-scoring one -- see that notebook's own step 6 for the full reasoning).

In [ ]:
import torch
import torch.nn as nn
import types

INPUT_SIZE = model.model.resolution


class ExportWrapper(nn.Module):
    '''Box-only export wrapper -- identical in spirit to train_object_detector.ipynb's
    own, minus the keypoint branch, since this model never predicts one (see step 4's
    markdown). raw_module still returns a trailing masks tensor; it's simply never
    referenced below -- see this step's markdown for why that alone keeps this export
    box-only with no mask-specific code needed at all.'''

    def __init__(self, raw_module, postprocess_module, input_size: int):
        super().__init__()
        self.raw_module = raw_module
        self.postprocess = postprocess_module
        self.input_size = input_size

    def forward(self, x):
        outputs_coord, outputs_class, *_ = self.raw_module(x)  # trailing masks tensor discarded
        out_dict = {"pred_logits": outputs_class, "pred_boxes": outputs_coord}
        target_sizes = torch.tensor([[self.input_size, self.input_size]], device=x.device)
        result = self.postprocess(out_dict, target_sizes)[0]  # batch size 1 -> one dict
        boxes = result["boxes"]
        scores = result["scores"].unsqueeze(-1)
        labels = result["labels"].unsqueeze(-1).float()
        return torch.cat([boxes, scores, labels], dim=-1)


def _select_topk_onnx_friendly(self, out_logits):
    '''Same ONNX-export patch as train_object_detector.ipynb -- see that notebook's own
    export step for why torch.argsort(..., stable=True) can't be exported.'''
    prob = out_logits.sigmoid()
    logits_for_topk = prob.view(out_logits.shape[0], -1)
    num_to_select = min(self.num_select, logits_for_topk.shape[1])
    topk_values, topk_indexes = torch.topk(logits_for_topk, num_to_select, dim=1)
    scores = topk_values
    topk_boxes = topk_indexes // out_logits.shape[2]
    labels = topk_indexes % out_logits.shape[2]
    return scores, labels, topk_boxes


raw_module = model.model.model

BEST_CKPT_PATH = Path("output") / "checkpoint_best_total.pth"
if BEST_CKPT_PATH.exists():
    from rfdetr.utilities.io import _safe_torch_load

    best_ckpt = _safe_torch_load(BEST_CKPT_PATH, trust=True)  # produced locally by this same run -- trusted
    raw_module.load_state_dict(best_ckpt["model"], strict=True)
    print(f"Loaded best checkpoint for export: {BEST_CKPT_PATH} (epoch {best_ckpt.get('epoch')})")
else:
    print(f"{BEST_CKPT_PATH} not found -- exporting the last trained epoch's weights instead.")

raw_module.export()   # swap in export-safe submodule implementations
raw_module.eval()
for p in raw_module.parameters():
    p.requires_grad = False

model.model.postprocess._select_topk = types.MethodType(_select_topk_onnx_friendly, model.model.postprocess)
model.model.postprocess.trace_alpha = 0.0  # disables keypoint-uncertainty fusion -- irrelevant here, harmless to set

wrapper = ExportWrapper(raw_module, model.model.postprocess, INPUT_SIZE)
wrapper.eval()
dummy_input = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE, device=next(raw_module.parameters()).device)

with torch.no_grad():
    torch.onnx.export(
        wrapper, dummy_input, "best.onnx",
        input_names=["images"], output_names=["output0"],
        opset_version=17,
        dynamo=False,  # the newer exporter needs onnxscript (not installed here) and can't trace this wrapper anyway
    )
print("Exported best.onnx (segmentation-trained, box-only output)")

## 6. Sanity-check the export (optional)

Same idea as the other two notebooks' own sanity checks -- confirms the output shape and that a real training image's top-scoring row looks like a real detection, not noise. Safe to skip if you'd rather just download `best.onnx` and test it directly in the app.

In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

sess = ort.InferenceSession("best.onnx", providers=["CPUExecutionProvider"])
img = Image.open(next((RAW_DIR / "images").glob("*.png"))).convert("RGB").resize((INPUT_SIZE, INPUT_SIZE))
blob = (np.array(img).astype(np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
output = sess.run(None, {"images": blob})[0]
print("output shape:", output.shape)  # (num_select, 6): x1,y1,x2,y2,score,label
top = output[np.argsort(-output[:, 4])[:5]]
print(top)

## 7. Download the trained model

**Slow on Colab?** `files.download()` below doesn't do a normal HTTP download -- it base64-encodes the whole file and pushes it through Colab's Python-kernel-to-browser message bridge, which is known to be slow for anything more than a few MB. If you've already mounted your Google Drive in this session, set `USE_GOOGLE_DRIVE = True` below to copy `best.onnx` there instead and download it from drive.google.com -- skips the slow bridge entirely.

In [ ]:
if ON_KAGGLE:
    # Kaggle captures every file left in /kaggle/working/ as this kernel's output -- no
    # equivalent of Colab's interactive download; cv_training.py's poll/pull step fetches
    # it afterward via `kaggle kernels output`.
    print("On Kaggle: best.onnx left in /kaggle/working/ -- fetched by the app's Kaggle poll/pull step.")
else:
    USE_GOOGLE_DRIVE = False  # set True if you've already mounted Drive this session -- see this cell's markdown

    if USE_GOOGLE_DRIVE:
        drive_dest = Path("/content/drive/MyDrive/best.onnx")
        if not drive_dest.parent.is_dir():
            raise RuntimeError(
                "Google Drive isn't mounted at /content/drive -- run "
                "`from google.colab import drive; drive.mount('/content/drive')` in a separate cell "
                "first (one-time permission prompt), then re-run this cell."
            )
        shutil.copy("best.onnx", drive_dest)
        print(f"Copied to Google Drive: {drive_dest} -- download it from drive.google.com.")
    else:
        from google.colab import files
        files.download("best.onnx")